# Play Tabular Pacman

Choose an environment with the buttons, preview an `rgb_array` frame inline, then run the last cell to play with the arrow keys in a pygame window.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
from PIL import Image
from masa.common.notebook_play import make_reset_env, sync_selected_env

VALID_ENV_NAMES = ("MiniPacman", "Pacman")

def make_env(env_name, **kwargs):
    if env_name == "MiniPacman":
        from masa.envs.tabular.mini_pacman import MiniPacman

        return MiniPacman(**kwargs)
    if env_name == "Pacman":
        from masa.envs.tabular.pacman import Pacman

        return Pacman(**kwargs)
    raise ValueError(f"env_name must be one of {VALID_ENV_NAMES!r}")

ENV_SELECTOR = widgets.ToggleButtons(
    options=VALID_ENV_NAMES,
    value="MiniPacman",
    description="Env",
)
display(ENV_SELECTOR)

ENV_NAME = ENV_SELECTOR.value
PACMAN_HAT = "cap"
GHOST_COLORS = ((238, 63, 88),)
SEED = 0


In [ ]:
ENV_NAME = ENV_SELECTOR.value
preview_env = make_env(
    ENV_NAME,
    render_mode="rgb_array",
    render_window_size=512,
    pacman_hat=PACMAN_HAT,
    ghost_colors=GHOST_COLORS,
)
preview_env.reset(seed=SEED)
frame = preview_env.render()
display(Image.fromarray(frame))
preview_env.close()


In [ ]:
def _make_env_kwargs():
    return {"pacman_hat": PACMAN_HAT, "ghost_colors": GHOST_COLORS}


def play(env_name=None, seed=SEED):
    import pygame

    follow_selector = env_name is None
    if follow_selector:
        env_name = ENV_SELECTOR.value
    action_keys = {
        pygame.K_LEFT: 0,
        pygame.K_RIGHT: 1,
        pygame.K_DOWN: 2,
        pygame.K_UP: 3,
        pygame.K_SPACE: 4,
    }
    env, _, _ = make_reset_env(
        make_env,
        env_name,
        seed=seed,
        render_mode="human",
        render_window_size=512,
        env_kwargs=_make_env_kwargs(),
    )

    try:
        running = True
        while running and not env.human_window_closed:
            if follow_selector:
                env, env_name, _, _, switched = sync_selected_env(
                    env,
                    env_name,
                    ENV_SELECTOR,
                    make_env,
                    seed=seed,
                    render_mode="human",
                    render_window_size=512,
                    env_kwargs=_make_env_kwargs(),
                    pygame=pygame,
                )
                if switched:
                    print("switched:", env_name)

            chosen_action = None
            for event in pygame.event.get():
                if not env.handle_pygame_event(event):
                    running = False
                    break
                if event.type == pygame.KEYDOWN:
                    if event.key in (pygame.K_q, pygame.K_ESCAPE):
                        running = False
                        break
                    chosen_action = action_keys.get(event.key, chosen_action)
            if chosen_action is None:
                env.render()
                continue
            _, reward, terminated, truncated, _ = env.step(chosen_action)
            if reward:
                print(f"reward={reward}")
            if terminated or truncated:
                env.reset(seed=seed)
    finally:
        env.close()

play()
